# ECABSD v3: Massive DIPS Training Pipeline
This notebook trains the ECABSD model on **17,770 massive protein graphs** from the DIPS dataset.

**Requirements:**
1. Select **GPU T4** runtime
2. Mount the Google Drive account containing `ECABSD_Massive_Dataset/`

The notebook automatically:
- Copies all data to local NVMe SSD (10x faster training)
- Creates a proper 85/15 train/val split
- Auto-detects node feature dimensions
- Launches training with full progress tracking

In [ ]:
# ==============================================================================
# 1. Mount Google Drive
# ⚠️ CRITICAL: When the popup appears, select your 5TB Google Account!
# ==============================================================================
import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

In [ ]:
# ==============================================================================
# 2. Clone Repository & Install Dependencies
# ==============================================================================
REPO = "/content/ecabsd_temp"
if not os.path.exists(REPO):
    !git clone https://github.com/nayanees6607/ecabsd_temp.git {REPO}

%cd {REPO}
!git checkout feature/massive-dataset-sota
!git pull

print("\nInstalling PyTorch Geometric (~2 mins)...")
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.3.0+cu121.html
!pip install -q biopython pydssp wandb

In [ ]:
# ==============================================================================
# 3. Copy Data to Local NVMe SSD (Multi-threaded, ~10 mins)
#    This is REQUIRED: training from Drive is 10x slower and compute_pos_weight
#    would take 2-4 hours loading files one-by-one over FUSE.
# ==============================================================================
import shutil, time
from concurrent.futures import ThreadPoolExecutor

DRIVE_DIR = "/content/drive/MyDrive/ECABSD_Massive_Dataset/data/dips_processed"
LOCAL_DIR = "/content/dips_local"

if not os.path.exists(LOCAL_DIR) or len(os.listdir(LOCAL_DIR)) < 1000:
    os.makedirs(LOCAL_DIR, exist_ok=True)

    print(f"Listing files in Google Drive...")
    all_files = [f for f in os.listdir(DRIVE_DIR) if f.endswith(".pt")]
    already = set(os.listdir(LOCAL_DIR))
    to_copy = [f for f in all_files if f not in already]
    print(f"Found {len(all_files)} total, {len(to_copy)} need copying")

    def copy_one(fname):
        shutil.copy2(os.path.join(DRIVE_DIR, fname), os.path.join(LOCAL_DIR, fname))
        return fname

    t0 = time.time()
    done = 0
    with ThreadPoolExecutor(max_workers=16) as pool:
        for _ in pool.map(copy_one, to_copy):
            done += 1
            if done % 500 == 0:
                elapsed = time.time() - t0
                rate = done / elapsed
                remaining = (len(to_copy) - done) / max(rate, 0.1)
                print(f"   Copied {done}/{len(to_copy)} ({rate:.1f} files/sec, ~{remaining:.0f}s left)")

    elapsed = time.time() - t0
    print(f"\n\u2705 Copy complete! {done} files in {elapsed:.0f}s ({done/max(elapsed,1):.1f} files/sec)")
else:
    local_count = len([f for f in os.listdir(LOCAL_DIR) if f.endswith(".pt")])
    print(f"\u2705 Local data already exists ({local_count} files). Skipping copy.")

In [ ]:
# ==============================================================================
# 4. Fix CSV (create 85/15 train/val split) & Auto-detect input_dim
# ==============================================================================
import csv, random, torch, yaml

CSV_SRC = "/content/drive/MyDrive/ECABSD_Massive_Dataset/data/dips_dataset.csv"
CSV_FIXED = "/content/dips_dataset_fixed.csv"

# Read CSV and strip Windows \r characters
with open(CSV_SRC, "r") as f:
    reader = list(csv.DictReader(f))

for row in reader:
    for key in row:
        if isinstance(row[key], str):
            row[key] = row[key].strip()

# Only keep rows whose chain_a .pt file exists locally
local_files = set(os.listdir(LOCAL_DIR))
valid_rows = []
for row in reader:
    fname_a = f"{row['pdb_id']}_{row['chain_a']}.pt"
    if fname_a in local_files:
        valid_rows.append(row)

print(f"CSV: {len(reader)} total entries, {len(valid_rows)} have matching .pt files")

# Shuffle and create 85/15 split
random.seed(42)
random.shuffle(valid_rows)
split_idx = int(len(valid_rows) * 0.85)
for i, row in enumerate(valid_rows):
    row["split"] = "train" if i < split_idx else "val"

with open(CSV_FIXED, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["pdb_id", "chain_a", "chain_b", "split"])
    writer.writeheader()
    for row in valid_rows:
        writer.writerow({k: row[k] for k in ["pdb_id", "chain_a", "chain_b", "split"]})

n_train = sum(1 for r in valid_rows if r["split"] == "train")
n_val   = sum(1 for r in valid_rows if r["split"] == "val")
print(f"\u2705 Fixed CSV: {n_train} train / {n_val} val")

# Auto-detect input_dim from actual data
pt_files = [f for f in os.listdir(LOCAL_DIR) if f.endswith(".pt")]
sample = torch.load(os.path.join(LOCAL_DIR, pt_files[0]), map_location="cpu", weights_only=False)
actual_dim = sample.x.shape[1]
print(f"\u2705 Detected input_dim = {actual_dim} from data")

# Write corrected config
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["processed_dir"] = LOCAL_DIR
cfg["data"]["splits_csv"]    = CSV_FIXED
cfg["model"]["input_dim"]    = actual_dim

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)

print(f"\n\ud83d\udccb Final Config:")
print(f"   processed_dir = {LOCAL_DIR}")
print(f"   splits_csv    = {CSV_FIXED}")
print(f"   input_dim     = {actual_dim}")
print(f"   hidden_dim    = {cfg['model']['hidden_dim']}")
print(f"   use_esm       = {cfg['model'].get('use_esm', False)}")
print(f"   gnn_type      = {cfg['model'].get('gnn_type', 'gcn')}")

In [ ]:
# ==============================================================================
# 5. Launch Training!
# ==============================================================================
%cd /content/ecabsd_temp

import sys
if '/content/ecabsd_temp' not in sys.path:
    sys.path.insert(0, '/content/ecabsd_temp')

print("\ud83d\ude80 Launching DIPS training on the branched model...\n")
from train import run_training
run_training()